# Understanding the Freddie Mac data

Freddie gives us **two kinds of file** per quarter, both pipe-delimited (`|`) with **no header row**:

| File | One row is... | Think of it as |
|---|---|---|
| `orig_YYYYQn.txt` | one loan | the loan's "birth certificate" (never changes) |
| `perf_YYYYQn.txt` | one loan in one month | the loan's monthly "report card" |

They are joined by `loan_seq` (the loan sequence number).

This notebook uses plain **pandas** on a small sample, so it runs on Windows without Spark or Java.

In [ ]:
%pip install pandas matplotlib

In [ ]:
from pathlib import Path
import pandas as pd

# The notebook lives in notebooks/, so the data is one folder up.
Q = "2023Q1"
folder = Path("..") / "data" / "raw" / "historical_data_2023" / f"historical_data_{Q}"
orig_file = folder / f"orig_{Q}.txt"
perf_file = folder / f"perf_{Q}.txt"
print(orig_file.exists(), perf_file.exists())

## 1. The loan file (one row per loan)

There is no header, so we supply the column names ourselves.
We name the first 24 columns, which we are confident about from the sample rows.
The last 7 are left as generic names until we confirm them against Freddie's official layout document.

In [ ]:
orig_cols = [
    "credit_score", "first_pay_date", "first_time_buyer", "maturity_date", "msa",
    "mi_pct", "num_units", "occupancy", "cltv", "dti", "orig_upb", "ltv",
    "int_rate", "channel", "ppm_flag", "amort_type", "state", "prop_type",
    "zip3", "loan_seq", "purpose", "orig_term", "num_borrowers", "seller",
    "col_25", "col_26", "col_27", "col_28", "col_29", "col_30", "col_31",
]

orig = pd.read_csv(orig_file, sep="|", header=None, names=orig_cols, dtype=str)

# Everything was read as text (safe). Convert the numeric columns we care about.
num_cols = ["credit_score", "cltv", "dti", "orig_upb", "ltv", "int_rate", "orig_term"]
orig[num_cols] = orig[num_cols].apply(pd.to_numeric, errors="coerce")

print(f"{len(orig):,} loans")
orig.head()

In [ ]:
# Quick summary of the numbers that will drive default / prepay behaviour.
orig[num_cols].describe().round(2)

Things to notice:
- `credit_score` of **9999** means "not available", not a real score.
- `orig_term` is the loan length in months (360 = 30 years, 240 = 20 years).
- Credit score, LTV (loan-to-value), DTI (debt-to-income) and interest rate are the classic drivers of default risk. Our step-4 hazard model will use them.

In [ ]:
# Loans per state (top 10) and per purpose. P=purchase, C=cash-out refi, N=no-cash-out refi
print(orig["state"].value_counts().head(10))
print()
print(orig["purpose"].value_counts())

## 2. The performance file (one row per loan per month)

This file is huge (about 700 MB for one quarter), so we only read the first 500,000 rows.
Names are given only for the columns we are sure of. The rest keep generic names.

In [ ]:
perf = pd.read_csv(perf_file, sep="|", header=None, dtype=str, nrows=500_000)

# Rename only the columns we are confident about (0-based positions).
known = {
    0: "loan_seq", 1: "report_period", 2: "current_upb", 3: "delinq_status",
    4: "loan_age", 5: "remaining_months", 7: "modification_flag",
    8: "zero_bal_code", 9: "zero_bal_date", 10: "current_rate",
}
perf = perf.rename(columns=known)
perf[["current_upb", "loan_age", "remaining_months", "current_rate"]] = perf[
    ["current_upb", "loan_age", "remaining_months", "current_rate"]
].apply(pd.to_numeric, errors="coerce")

print(f"{len(perf):,} rows,  {perf['loan_seq'].nunique():,} distinct loans,  {perf.shape[1]} columns")
perf[list(known.values())].head()

## 3. Follow one loan through time

This is the core idea of the whole project. Our simulation will produce a table like this for **every loan, every month, under every scenario**.

Watch the balance fall (the borrower pays down the loan) and `remaining_months` count down.

In [ ]:
# Pick a loan with a long history (many monthly rows) so the story is visible.
loan_id = perf["loan_seq"].value_counts().index[0]
one = perf[perf["loan_seq"] == loan_id].sort_values("report_period")

print(loan_id, f"- {len(one)} months")
one[["report_period", "loan_age", "remaining_months", "current_upb", "delinq_status", "zero_bal_code"]].head(12)

In [ ]:
one.plot(x="loan_age", y="current_upb", title=f"Balance of {loan_id} over time", legend=False);

## 4. How loans go bad or end

`delinq_status` is the number of months behind on payments (`00` = current, `01` = 30 days late, `02` = 60 days ...).
`zero_bal_code` is filled in only in the **last** row of a loan that has ended. `01` means prepaid or paid off, and codes like `02`, `03`, `09` mean the loan ended in a loss event.

These two columns are what we will use to **calibrate the prepay / default / pay hazards** in step 4.

In [ ]:
print("Delinquency status (share of loan-months):")
print(perf["delinq_status"].value_counts(normalize=True).head(6).round(4))
print()
print("How loans ended (zero_bal_code, only present on final rows):")
print(perf["zero_bal_code"].value_counts(dropna=False))

## 5. Why this becomes a scale problem

For each loan, the simulation must roll forward `remaining_months` more months, once per scenario.

In [ ]:
# Latest row per loan = where the simulation would start.
latest = perf.sort_values("report_period").groupby("loan_seq").tail(1)
active = latest[latest["zero_bal_code"].isna()]

scenarios = 3
cells = active["remaining_months"].sum() * scenarios
print(f"Active loans in this sample: {len(active):,}")
print(f"Simulation cells (loan x month x scenario): {cells:,.0f}")
print(f"Scaled to 55M loans: roughly {cells / len(active) * 55_000_000:,.0f} cells")